# 260504 promotion split

`Membership_v2.csv`의 `is_promotion` 값을 기준으로 데이터를 2개 집단으로 분리한다.

- `promotion_1`: `is_promotion == 1`인 멤버십 행에 등장한 `USER_KEY` 기준
- `promotion_0`: `is_promotion == 0`인 멤버십 행에 등장한 `USER_KEY` 기준
- `View_History_v2.csv`는 `User_Mapping_v2.csv`를 거쳐 `USER_NUM` 기준으로 필터링한다.
- `Movie_Master_v2.csv`는 각 집단의 시청 이력에 등장한 `MOVIE_NUM`만 남긴다.

In [45]:
from pathlib import Path

import pandas as pd


SOURCE_DIR = Path.cwd().parent / "260504_view history delete"
OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

membership = pd.read_csv(SOURCE_DIR / "Membership_v2.csv")
user_mapping = pd.read_csv(SOURCE_DIR / "User_Mapping_v2.csv")
view_history = pd.read_csv(SOURCE_DIR / "View_History_v2.csv")
movie_master = pd.read_csv(SOURCE_DIR / "Movie_Master_v2.csv")

print("membership:   ", membership.shape)
print("user_mapping: ", user_mapping.shape)
print("view_history: ", view_history.shape)
print("movie_master: ", movie_master.shape)

membership:    (14873, 15)
user_mapping:  (14892, 2)
view_history:  (106205, 5)
movie_master:  (14018, 3)


In [46]:
print("is_promotion 분포:")
print(membership["is_promotion"].value_counts(dropna=False))

is_promotion 분포:
is_promotion
1    7550
0    7323
Name: count, dtype: int64


In [47]:
def split_by_promotion(is_promotion: int) -> dict[str, pd.DataFrame]:
    membership_split = membership.loc[
        membership["is_promotion"].eq(is_promotion)
    ].copy()

    user_keys = membership_split["USER_KEY"].dropna().unique()

    user_mapping_split = user_mapping.loc[
        user_mapping["USER_KEY"].isin(user_keys)
    ].copy()

    user_nums = user_mapping_split["USER_NUM"].dropna().unique()

    view_history_split = view_history.loc[
        view_history["USER_NUM"].isin(user_nums)
    ].copy()

    movie_nums = view_history_split["MOVIE_NUM"].dropna().unique()

    movie_master_split = movie_master.loc[
        movie_master["MOVIE_NUM"].isin(movie_nums)
    ].copy()

    return {
        "membership": membership_split,
        "user_mapping": user_mapping_split,
        "view_history": view_history_split,
        "movie_master": movie_master_split,
    }


promotion_1 = split_by_promotion(1)
promotion_0 = split_by_promotion(0)

summary = []
for group_name, group_data in {"promotion_1": promotion_1, "promotion_0": promotion_0}.items():
    for data_name, data in group_data.items():
        summary.append({"group": group_name, "data": data_name, "rows": len(data), "columns": len(data.columns)})

pd.DataFrame(summary)

,group,data,rows,columns
0,promotion_1,membership,7550,15
1,promotion_1,user_mapping,7558,2
2,promotion_1,view_history,54370,5
3,promotion_1,movie_master,4003,3
4,promotion_0,membership,7323,15
5,promotion_0,user_mapping,7176,2
6,promotion_0,view_history,50858,5
7,promotion_0,movie_master,3874,3


In [48]:
output_map = {
    "promotion_1_membership_v2.csv":   promotion_1["membership"],
    "promotion_1_user_mapping_v2.csv": promotion_1["user_mapping"],
    "promotion_1_view_history_v2.csv": promotion_1["view_history"],
    "promotion_1_movie_master_v2.csv": promotion_1["movie_master"],
    "promotion_0_membership_v2.csv":   promotion_0["membership"],
    "promotion_0_user_mapping_v2.csv": promotion_0["user_mapping"],
    "promotion_0_view_history_v2.csv": promotion_0["view_history"],
    "promotion_0_movie_master_v2.csv": promotion_0["movie_master"],
}

for file_name, data in output_map.items():
    output_path = OUTPUT_DIR / file_name
    data.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"saved: {output_path.name}  rows={len(data):,}")

saved: promotion_1_membership_v2.csv  rows=7,550
saved: promotion_1_user_mapping_v2.csv  rows=7,558
saved: promotion_1_view_history_v2.csv  rows=54,370
saved: promotion_1_movie_master_v2.csv  rows=4,003
saved: promotion_0_membership_v2.csv  rows=7,323
saved: promotion_0_user_mapping_v2.csv  rows=7,176
saved: promotion_0_view_history_v2.csv  rows=50,858
saved: promotion_0_movie_master_v2.csv  rows=3,874


In [49]:
promotion_1_user_keys = set(promotion_1["membership"]["USER_KEY"])
promotion_0_user_keys = set(promotion_0["membership"]["USER_KEY"])
overlap_user_keys = promotion_1_user_keys & promotion_0_user_keys

print("promotion_1 USER_KEY 수:", len(promotion_1_user_keys))
print("promotion_0 USER_KEY 수:", len(promotion_0_user_keys))
print("두 집단에 모두 등장하는 USER_KEY 수:", len(overlap_user_keys))

promotion_1 USER_KEY 수: 7550
promotion_0 USER_KEY 수: 7140
두 집단에 모두 등장하는 USER_KEY 수: 59


In [50]:
# 겹치는 USER_KEY의 Membership 행 확인
overlap_rows = membership.loc[
    membership["USER_KEY"].isin(overlap_user_keys)
].sort_values(["USER_KEY", "is_promotion"])

print(f"겹치는 USER_KEY {len(overlap_user_keys)}명의 Membership 행 수: {len(overlap_rows)}")
print("→ 동일 USER_KEY가 is_promotion=1 행과 is_promotion=0 행을 모두 가짐\n")
overlap_rows[["USER_KEY", "is_promotion", "reg_date", "end_date"]]

겹치는 USER_KEY 59명의 Membership 행 수: 118
→ 동일 USER_KEY가 is_promotion=1 행과 is_promotion=0 행을 모두 가짐



,USER_KEY,is_promotion,reg_date,end_date
10689,01b73db46b0b3f6c24ca22dc4629ed41ed0d6ac4beccf8...,0,2021-03-14,2021-04-14
10212,01b73db46b0b3f6c24ca22dc4629ed41ed0d6ac4beccf8...,1,2021-03-14,2021-03-14
8393,06d71531aa38d985f29324bb382df1c5846737e19bc9a6...,0,2021-03-12,2021-04-12
8915,06d71531aa38d985f29324bb382df1c5846737e19bc9a6...,1,2021-03-12,2021-03-12
11191,06e29aff8528056461562d5763bded5505f51c9b893148...,0,2021-03-06,2021-04-06
...,...,...,...,...
14843,f06951ff35605cfa7796e238a35482c0593a7dc566ef49...,1,2021-03-13,2021-03-14
13544,f4aff9b4a4770e81f39fb54abda4f3c183faf05bf78077...,0,2021-03-11,2021-04-11
12991,f4aff9b4a4770e81f39fb54abda4f3c183faf05bf78077...,1,2021-03-11,2021-03-11
11888,f8fc450e642cb1f09534747f5f5630f3ae5ba7c05c16ae...,0,2021-03-14,2021-04-14
